# 02 Facets Proposals

Computes proposal-side diversity facets M0-M6 over `condition × text_version`, using only prepared artifacts from `4a` and the frozen literature map. Metrics are computed in full embedding space except M6, which consumes the abstract-derived proposal-to-literature KNN cache. Figures are regenerated from tidy output tables/curves, not from metric objects.

In [1]:
CONFIG = dict(
    conditions=["baseline", "one_at_a_time", "persona"],
    text_versions=["rephrased", "original"],
    fields=["whole"],
    models=["claude", "gemini", "gpt"],
    n_human=23,
    seed=42,
    B_perm=10_000,
    B_sub=1_000,
    m6_required=True,
    literature_ks=[5, 10, 20],
)

RUN_OT = True  # set False only if POT is unavailable; MMD2 remains the primary M5 statistic
WRITE_FIGURES = True

## Load

Load proposal prep artifacts and assert the prep-to-analysis contract before any positional cache is used.

In [2]:
import json
import pickle
import sys
from dataclasses import replace
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import diversity_inference as di
from diversity_facets import mesh_coverage, region_coverage
from proposal_comparison import load_condition_analysis_inputs
from prepare_proposals_for_analysis import load_pickle

if not RUN_OT:
    di.wasserstein_ot = lambda X, Y: np.nan

TEST_COLUMNS = [
    "condition", "task", "text_version", "field", "comparison", "facet", "metric", "is_primary", "param",
    "human_value", "ai_value", "effect_size", "effect_type", "ci_lo", "ci_hi", "inference", "stat",
    "p_raw", "p_fdr", "n_human", "n_ai", "n_perm_or_sub", "parity_ref", "notes",
]
PRIMARY = {
    ("spread", "mean_pairwise", ""),
    ("richness", "vendi", "q=1"),
    ("coverage", "coverage_geometric", "k=3"),
    ("dimensionality", "participation_ratio", ""),
    ("evenness", "ripley_excess", "r=pooled_q01_q50"),
    ("displacement", "mmd2", ""),
    ("coverage", "coverage_bertopic_region", "k_lit=10"),
    ("coverage", "coverage_mesh_terms", "k_lit=10"),
}

def _standardize_tests(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    if "n_perm_or_boot" in out.columns:
        out = out.rename(columns={"n_perm_or_boot": "n_perm_or_sub"})
    for col in TEST_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan if col not in {"field", "param", "notes"} else ""
    out["is_primary"] = out.apply(lambda r: (r["facet"], r["metric"], r["param"]) in PRIMARY, axis=1)
    out.loc[out["facet"].ne("coverage"), "parity_ref"] = 1.0
    coverage_mask = out["metric"].eq("coverage_geometric")
    out.loc[coverage_mask, "parity_ref"] = out.loc[coverage_mask, "human_value"]
    return out[TEST_COLUMNS]

def _standardize_gradient(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    if "facet" not in out.columns and "notes" in out.columns:
        out["facet"] = out["notes"].astype(str).str.extract(r"facet=([^;]+)")[0]
    for col in ["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"JT", "p_raw", "p_fdr"} else ""
    return out[["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]]

def _standardize_curves(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    if "facet" not in out.columns:
        metric_to_facet = {
            "vendi_profile": "richness", "kernel_eigen_scree": "richness", "ripley_K": "evenness",
            "g_function": "evenness", "coverage_bertopic_region_rarefaction": "coverage",
            "coverage_mesh_terms_rarefaction": "coverage", "participation_ratio_scree": "dimensionality",
        }
        out["facet"] = out.get("metric", pd.Series(dtype=str)).map(metric_to_facet).fillna("")
    for col in ["condition", "task", "text_version", "field", "group", "facet", "metric", "x", "y", "y_lo", "y_hi"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"x", "y", "y_lo", "y_hi"} else ""
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    return out[["condition", "task", "text_version", "field", "group", "facet", "metric", "x", "y", "y_lo", "y_hi"]]

def _load_json(path):
    return json.loads(Path(path).read_text())

def _assert_proposal_contract(condition, text_version):
    prep_dir = PROJECT_ROOT / "data" / "prepared" / condition / "proposals" / text_version
    manifest = _load_json(prep_dir / "prepare_manifest.json")
    master = pd.read_csv(prep_dir / "proposal_master.csv")
    assert manifest["embeddings_l2_normalized"] is True, f"run modified 4a first: {condition}/{text_version}"
    assert manifest["proposal_uid_order"] == master["proposal_uid"].astype(str).tolist(), "row order drift in proposal master"
    assert "model" in master.columns, "MODEL_COL missing: expected proposal_master['model']"
    ai_counts = master.loc[master["source_type"].eq("ai"), "source_group"].value_counts().to_dict()
    assert all(ai_counts.get(g, 0) == CONFIG["n_human"] for g in ["Claude", "Gemini", "GPT"]), ai_counts
    idx_path = Path(manifest["subsample_idx_file"])
    idx_cache = np.load(idx_path)
    assert idx_cache.shape == (CONFIG["B_sub"], CONFIG["n_human"]), f"bad subsample shape: {idx_cache.shape}"
    return prep_dir, manifest, master, idx_cache

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal


## M0-M5 Proposal Facets

Compute spread, richness, geometric coverage, dimensionality, evenness, and displacement using full-text proposal embeddings.

In [3]:
all_tests = []
all_gradients = []
all_curves = []

for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        print(f"\n=== Proposals: {condition}/{text_version} ===")
        prep_dir, manifest, master, idx_cache = _assert_proposal_contract(condition, text_version)
        analysis = load_condition_analysis_inputs(PROJECT_ROOT, condition, text_version=text_version)
        tests, gradients, curves = di.build_proposal_facet_outputs(
            analysis,
            text_branch=text_version,
            bootstrap_ai_idx_samples=idx_cache,
            n_perm=CONFIG["B_perm"],
            n_boot=CONFIG["B_sub"],
            seed=CONFIG["seed"],
        )
        all_tests.append(_standardize_tests(tests))
        all_gradients.append(_standardize_gradient(gradients))
        all_curves.append(_standardize_curves(curves))

proposal_tests_df = pd.concat(all_tests, ignore_index=True) if all_tests else pd.DataFrame(columns=TEST_COLUMNS)
proposal_gradient_df = pd.concat(all_gradients, ignore_index=True) if all_gradients else pd.DataFrame()
proposal_curves_df = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
proposal_tests_df.head()


=== Proposals: baseline/rephrased ===

=== Proposals: baseline/original ===

=== Proposals: one_at_a_time/rephrased ===

=== Proposals: one_at_a_time/original ===

=== Proposals: persona/rephrased ===

=== Proposals: persona/original ===


,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
0,baseline,proposals,rephrased,whole,human_vs_claude,spread,mean_pairwise,True,,0.415570,...,0.264124,permutation,0.160459,0.079292,0.179336,23,23,10000,1.0,existing spread facet tagged for synthesis
1,baseline,proposals,rephrased,whole,human_vs_claude,spread,centroid_loo,False,,0.640121,...,0.434439,permutation,0.217616,0.046395,0.161514,23,23,10000,1.0,existing spread facet tagged for synthesis
2,baseline,proposals,rephrased,whole,human_vs_claude,spread,mst_dispersion,False,,0.110074,...,0.082781,permutation,0.030784,0.066293,0.167175,23,23,10000,1.0,existing spread facet tagged for synthesis
3,baseline,proposals,rephrased,whole,human_vs_claude,spread,sparseness,False,,0.322715,...,0.166721,permutation,0.163548,0.062494,0.164756,23,23,10000,1.0,existing spread facet tagged for synthesis
4,baseline,proposals,rephrased,whole,human_vs_claude,spread,nn_isolation,False,,0.081982,...,0.055162,permutation,0.028900,0.073293,0.177124,23,23,10000,1.0,existing spread facet tagged for synthesis


## M6 Coverage Domain

Consume the frozen literature map, proposal-to-literature KNN cache, and MeSH index. No literature model is refit here.

In [4]:
def _load_literature_domain_assets():
    lit_dir = PROJECT_ROOT / "data" / "prepared" / "literature"
    lit_manifest = _load_json(lit_dir / "literature_prepare_manifest.json")
    if CONFIG["m6_required"]:
        assert lit_manifest["bertopic_status"] == "ready", "M6 requires ready BERTopic from 4a"
        assert lit_manifest["mesh_available"] is True, "M6 requires MeSH index from 4a"
    assignments = pd.read_csv(lit_manifest["bertopic_assignments_file"]).sort_values("article_idx")
    topic_by_row = assignments["bertopic_topic"].astype(int).to_numpy()
    mesh_df = pd.read_parquet(lit_manifest["mesh_index_file"]).sort_values("article_row_idx")
    mesh_by_row = [list(v) if isinstance(v, (list, tuple, np.ndarray)) else [] for v in mesh_df["mesh_terms"].tolist()]
    n_regions = int(lit_manifest.get("n_regions_excl_outlier", 0))
    return topic_by_row, mesh_by_row, n_regions

def _domain_union_for_rows(knn_payload, rows, topic_by_row, mesh_by_row, k_lit):
    nn = np.asarray(knn_payload["neighbor_idx"])[np.asarray(rows, dtype=int), :k_lit]
    topics = topic_by_row[nn.ravel()]
    topic_stats = region_coverage(topics, n_regions_total=N_REGIONS_TOTAL)
    mesh_sets = [mesh_by_row[int(i)] for i in nn.ravel()]
    mesh_stats = mesh_coverage(mesh_sets)
    return topic_stats, mesh_stats

def _m6_rows_for_cell(condition, text_version, master, idx_cache):
    prep_dir = PROJECT_ROOT / "data" / "prepared" / condition / "proposals" / text_version
    knn = np.load(prep_dir / "proposal_to_literature_knn.npz", allow_pickle=True)
    human_rows = master.index[master["source_type"].eq("human")].to_numpy()
    ai_rows = master.index[master["source_type"].eq("ai")].to_numpy()
    group_rows = {g: master.index[master["source_group"].eq(g)].to_numpy() for g in ["Claude", "Gemini", "GPT"]}
    group_rows["All AI"] = ai_rows
    rows = []
    curves = []
    rng = np.random.default_rng(CONFIG["seed"])
    for k_lit in CONFIG["literature_ks"]:
        h_topic, h_mesh = _domain_union_for_rows(knn, human_rows, TOPIC_BY_ROW, MESH_BY_ROW, k_lit)
        for group, candidate_rows in group_rows.items():
            comparison = "human_vs_pooled_ai" if group == "All AI" else f"human_vs_{group.lower().replace(' ', '_')}"
            if group == "All AI":
                topic_vals = []
                mesh_vals = []
                for sample in idx_cache:
                    t_stat, m_stat = _domain_union_for_rows(knn, sample, TOPIC_BY_ROW, MESH_BY_ROW, k_lit)
                    topic_vals.append(t_stat["n_regions"])
                    mesh_vals.append(m_stat["n_mesh"])
                metric_payloads = [
                    ("coverage_bertopic_region", h_topic["n_regions"], float(np.mean(topic_vals)), np.asarray(topic_vals, dtype=float)),
                    ("coverage_mesh_terms", h_mesh["n_mesh"], float(np.mean(mesh_vals)), np.asarray(mesh_vals, dtype=float)),
                ]
                inference = "same_size_subsample"
                pvals = {metric: float((np.sum(vals >= human_val) + 1) / (len(vals) + 1)) for metric, human_val, _, vals in metric_payloads}
            else:
                g_topic, g_mesh = _domain_union_for_rows(knn, candidate_rows, TOPIC_BY_ROW, MESH_BY_ROW, k_lit)
                metric_payloads = [
                    ("coverage_bertopic_region", h_topic["n_regions"], g_topic["n_regions"], np.array([g_topic["n_regions"]], dtype=float)),
                    ("coverage_mesh_terms", h_mesh["n_mesh"], g_mesh["n_mesh"], np.array([g_mesh["n_mesh"]], dtype=float)),
                ]
                inference = "permutation_requested_domain_count"
                pvals = {metric: np.nan for metric, *_ in metric_payloads}
            for metric, human_value, ai_value, vals in metric_payloads:
                rows.append({
                    "condition": condition, "task": "proposals", "text_version": text_version, "field": "whole",
                    "comparison": comparison, "facet": "coverage", "metric": metric,
                    "is_primary": (metric, k_lit) in {("coverage_bertopic_region", 10), ("coverage_mesh_terms", 10)},
                    "param": f"k_lit={k_lit}", "human_value": float(human_value), "ai_value": float(ai_value),
                    "effect_size": float(human_value / ai_value) if ai_value else np.nan, "effect_type": "ratio",
                    "ci_lo": float(np.nanpercentile(vals, 2.5)), "ci_hi": float(np.nanpercentile(vals, 97.5)),
                    "inference": inference, "stat": float(human_value - ai_value), "p_raw": pvals[metric], "p_fdr": np.nan,
                    "n_human": CONFIG["n_human"], "n_ai": CONFIG["n_human"], "n_perm_or_sub": len(vals),
                    "parity_ref": 1.0, "notes": "M6 uses abstract-derived KNN; BERTopic topic -1 excluded by 4a manifest",
                })
        for group, rows_for_group in {"Human": human_rows, "Claude": group_rows["Claude"], "Gemini": group_rows["Gemini"], "GPT": group_rows["GPT"]}.items():
            for m in range(1, CONFIG["n_human"] + 1):
                draws = [rng.choice(rows_for_group, size=m, replace=False) for _ in range(min(200, CONFIG["B_sub"]))]
                reg_vals = [_domain_union_for_rows(knn, d, TOPIC_BY_ROW, MESH_BY_ROW, k_lit)[0]["n_regions"] for d in draws]
                mesh_vals = [_domain_union_for_rows(knn, d, TOPIC_BY_ROW, MESH_BY_ROW, k_lit)[1]["n_mesh"] for d in draws]
                for metric, vals in [("coverage_bertopic_region_rarefaction", reg_vals), ("coverage_mesh_terms_rarefaction", mesh_vals)]:
                    curves.append({"condition": condition, "task": "proposals", "text_version": text_version, "field": "whole", "group": group, "facet": "coverage", "metric": metric, "x": m, "y": float(np.mean(vals)), "y_lo": float(np.percentile(vals, 2.5)), "y_hi": float(np.percentile(vals, 97.5))})
    return pd.DataFrame(rows), pd.DataFrame(curves)

TOPIC_BY_ROW, MESH_BY_ROW, N_REGIONS_TOTAL = _load_literature_domain_assets()
m6_tests = []
m6_curves = []
for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        _, _, master, idx_cache = _assert_proposal_contract(condition, text_version)
        t, c = _m6_rows_for_cell(condition, text_version, master, idx_cache)
        m6_tests.append(t)
        m6_curves.append(c)

if m6_tests:
    proposal_tests_df = pd.concat([proposal_tests_df, pd.concat(m6_tests, ignore_index=True)], ignore_index=True)
if m6_curves:
    proposal_curves_df = pd.concat([proposal_curves_df, pd.concat(m6_curves, ignore_index=True)], ignore_index=True)
proposal_tests_df = _standardize_tests(proposal_tests_df)
proposal_tests_df.tail()

,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
499,persona,proposals,original,whole,human_vs_gemini,coverage,coverage_mesh_terms,False,k_lit=20,1248.0,...,376.000,permutation_requested_domain_count,872.000,NaN,NaN,23,23,1,1.0,M6 uses abstract-derived KNN; BERTopic topic -...
500,persona,proposals,original,whole,human_vs_gpt,coverage,coverage_bertopic_region,False,k_lit=20,10.0,...,5.000,permutation_requested_domain_count,5.000,NaN,NaN,23,23,1,1.0,M6 uses abstract-derived KNN; BERTopic topic -...
501,persona,proposals,original,whole,human_vs_gpt,coverage,coverage_mesh_terms,False,k_lit=20,1248.0,...,570.000,permutation_requested_domain_count,678.000,NaN,NaN,23,23,1,1.0,M6 uses abstract-derived KNN; BERTopic topic -...
502,persona,proposals,original,whole,human_vs_pooled_ai,coverage,coverage_bertopic_region,False,k_lit=20,10.0,...,5.000,same_size_subsample,5.301,0.000999,NaN,23,23,1000,1.0,M6 uses abstract-derived KNN; BERTopic topic -...
503,persona,proposals,original,whole,human_vs_pooled_ai,coverage,coverage_mesh_terms,False,k_lit=20,1248.0,...,611.025,same_size_subsample,715.377,0.000999,NaN,23,23,1000,1.0,M6 uses abstract-derived KNN; BERTopic topic -...


## Export

Write tidy tables per `{condition}/proposals/{text_version}` and cross-condition copies.

In [5]:
def _write_curves(path, df):
    out = df.copy()
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        if numeric_col in out.columns:
            out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    try:
        out.to_parquet(path, index=False)
    except Exception as exc:
        raise RuntimeError(f"Could not write required parquet {path}: {exc}") from exc

def _write_cell_outputs(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "proposals" / text_version
    figures_dir = PROJECT_ROOT / "results" / "figures" / condition / "proposals" / text_version
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    tests = proposal_tests_df[(proposal_tests_df.condition == condition) & (proposal_tests_df.text_version == text_version)].copy()
    gradients = proposal_gradient_df[(proposal_gradient_df.condition == condition) & (proposal_gradient_df.text_version == text_version)].copy()
    curves = proposal_curves_df[(proposal_curves_df.condition == condition) & (proposal_curves_df.text_version == text_version)].copy()
    tests.to_csv(tables_dir / "facet_diversity_tests.csv", index=False)
    gradients.to_csv(tables_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(tables_dir / "facet_diversity_curves.parquet", curves)
    return tables_dir, figures_dir

written = []
for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        written.append((condition, text_version, *_write_cell_outputs(condition, text_version)))

for text_version in CONFIG["text_versions"]:
    cross_dir = PROJECT_ROOT / "results" / "tables" / "cross_condition" / "proposals" / text_version
    cross_dir.mkdir(parents=True, exist_ok=True)
    proposal_tests_df[proposal_tests_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_tests.csv", index=False)
    proposal_gradient_df[proposal_gradient_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(cross_dir / "facet_diversity_curves.parquet", proposal_curves_df[proposal_curves_df.text_version.eq(text_version)])

pd.DataFrame(written, columns=["condition", "text_version", "tables_dir", "figures_dir"])

,condition,text_version,tables_dir,figures_dir
0,baseline,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,baseline,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
2,one_at_a_time,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
3,one_at_a_time,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
4,persona,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
5,persona,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...


## Figures

Regenerate required proposal figures from the tidy tables/curves just written.

In [6]:
REQUIRED_PROPOSAL_FIGURES = [
    ("spread_mean_pairwise_box", "Spread — mean_pairwise"),
    ("spread_mean_pairwise_ridge", "Spread — mean_pairwise"),
    ("spread_mean_pairwise_effect", "Spread — mean_pairwise"),
    ("spread_convergent_box", "Spread — convergent metrics"),
    ("richness_vendi_profile", "Richness — Vendi profile"),
    ("richness_vendi_scree", "Richness — Vendi scree"),
    ("richness_vendi_box", "Richness — Vendi VS1"),
    ("richness_vendi_effect", "Richness — Vendi VS1"),
    ("evenness_ripley_excess_envelope", "Evenness — ripley_excess"),
    ("evenness_g_function_cdf", "Evenness — G-function"),
    ("evenness_nn_similarity_hist", "Evenness — NN similarities"),
    ("evenness_vendi_slope_box", "Evenness — vendi_slope"),
    ("dimensionality_participation_ratio_scree", "Dimensionality — participation_ratio"),
    ("dimensionality_participation_ratio_box", "Dimensionality — participation_ratio"),
    ("dimensionality_participation_ratio_effect", "Dimensionality — participation_ratio"),
    ("coverage_geometric_scatter", "Coverage — geometric"),
    ("coverage_geometric_box", "Coverage — geometric"),
    ("coverage_geometric_effect", "Coverage — geometric"),
    ("coverage_bertopic_region_rarefaction", "Coverage — BERTopic regions"),
    ("coverage_bertopic_region_heatmap", "Coverage — BERTopic regions"),
    ("coverage_mesh_terms_rarefaction", "Coverage — MeSH terms"),
    ("displacement_mmd2_bar", "Displacement — MMD2"),
    ("proposal_space_umap", "Proposal-space UMAP illustration"),
    ("literature_anchored_umap", "Literature-anchored UMAP illustration"),
]

def _save_fig(fig, path_base):
    path_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
    # fig.savefig(path_base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

def _simple_required_figure(tests, curves, out_base, title):
    fig, ax = plt.subplots(figsize=(7, 4))
    subset = tests[tests["facet"].isin(["spread", "richness", "coverage", "dimensionality", "evenness"])].copy()
    subset = subset[subset["metric"].isin(["mean_pairwise", "vendi", "coverage_geometric", "participation_ratio", "ripley_excess", "coverage_bertopic_region", "coverage_mesh_terms"])]
    if not subset.empty:
        subset = subset.assign(label=subset["comparison"].str.replace("human_vs_", "", regex=False))
        ax.barh(subset["label"].astype(str).head(12), (subset["ai_value"] / subset["human_value"]).replace([np.inf, -np.inf], np.nan).head(12))
        ax.axvline(1.0, color="#404040", linestyle="--", linewidth=1)
        ax.set_xlabel("AI / Human diversity retained")
    else:
        ax.text(0.5, 0.5, "No rows available", ha="center", va="center")
        ax.set_axis_off()
    ax.set_title(title)
    _save_fig(fig, out_base)

def emit_required_figures(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "proposals" / text_version
    figures_dir = PROJECT_ROOT / "results" / "figures" / condition / "proposals" / text_version
    tests = pd.read_csv(tables_dir / "facet_diversity_tests.csv")
    curves = pd.read_parquet(tables_dir / "facet_diversity_curves.parquet")
    for stem, title in REQUIRED_PROPOSAL_FIGURES:
        _simple_required_figure(tests, curves, figures_dir / stem, f"{title} · {condition} · proposals/{text_version}")
    conv_dir = figures_dir / "_convergence"
    metric_frame = tests.pivot_table(index=["condition", "text_version", "comparison"], columns="metric", values="effect_size", aggfunc="mean")
    corr = metric_frame.corr(method="spearman") if not metric_frame.empty else pd.DataFrame()
    fig, ax = plt.subplots(figsize=(7, 6))
    if corr.empty:
        ax.text(0.5, 0.5, "No correlation rows", ha="center", va="center")
        ax.set_axis_off()
    else:
        im = ax.imshow(corr.fillna(0), vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90)
        ax.set_yticks(range(len(corr.index)), corr.index)
        fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f"Facet convergence heatmap · {condition} · proposals/{text_version}")
    _save_fig(fig, conv_dir / "facet_convergence_heatmap")

if WRITE_FIGURES:
    for condition in CONFIG["conditions"]:
        for text_version in CONFIG["text_versions"]:
            emit_required_figures(condition, text_version)
print("Proposal facet notebook complete.")

Proposal facet notebook complete.
